# Process APT experiment

## Label the dataset
For each experiment, there is a dedicated folder containing:
- .csv converted from .pcap file, containing extracted flows.
- .pkl contains time intervals in which attacks are performed (and additional info)
- .npz contains datetime objects that mark the start of the experiment.

In [ ]:
# Make project helpers available from a kernel started in this folder or the repo root.
from pathlib import Path
import sys
PROJECT_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents)
                    if (root / "preprocessing").is_dir() and (root / "scenario_reconstructor").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from datetime import datetime
import pytz
import pickle

In [ ]:
root = "./data/dollar_char_train/"
csv_file = root + "dollar_char_train.csv"
intervals = root + "dollar_char_train.pkl"
start = root + "dollar_char_train.npz"

### Load experiment timestamps
We load the timestamps indicating the start and the end of each attacker phase/step.

In [ ]:
import pickle
with open(intervals, 'rb') as f:
    loaded_data = pickle.load(f)

In [ ]:
len(loaded_data)

In [ ]:
loaded_data

### Load CSV network traffic

In [ ]:
df = pd.read_csv(csv_file)

In [ ]:
df.shape

In [ ]:
list(df.columns)

In [ ]:
df['bidirectional_first_seen_ms']

### Load experiment timestamps
We load the timestamps indicating the start and the end of each attacker phase/step.

In [ ]:
import pickle
with open(intervals, 'rb') as f:
    loaded_data = pickle.load(f)

In [ ]:
len(loaded_data)

In [ ]:
loaded_data

### Count sequences for attack type

In [ ]:
from collections import Counter
command_counts = Counter(item['command'] for item in loaded_data)
print(command_counts)

Load start_time FROM .npy file. This is the starting timestamp of the apt simulation.

### Uniform time zone for start timestamp and experiment timestamps

In [ ]:
start_time = np.load(start, allow_pickle=True)
start_time = start_time['datetime']

In [ ]:
start_time

In [ ]:
from preprocessing.time import shift_capture_time
start_time = shift_capture_time(start_time.item())

In [ ]:
start_time

### Convert relative timestamps to UTC

We convert the 'layers.frame.frame.time_epoch' relative time to absolute UTC timestamps using as reference the start_time we load from the .npy file.

In [ ]:
from preprocessing.time import absolute_flow_time
convert_to_absolute_time = absolute_flow_time
df['timestamp'] = df['bidirectional_first_seen_ms'].apply(convert_to_absolute_time, start_time=start_time)

In [ ]:
df['timestamp'], df['bidirectional_first_seen_ms']

### Assign labels
- We assign labels based on phase name:  if observation is in the start_time and end_time of a phase is marked with the corresponding phase name.
- Not labeled observations are marked as "discard": they correspond sleep times between attacker steps/technique. 

In [ ]:
import numpy as np
import pandas as pd

ph = pd.DataFrame(loaded_data)

print(ph) 

In [ ]:
ph[0:10]

In [ ]:
ph['start_time'] = pd.to_datetime(ph['start_time'], utc=True)
ph['end_time']   = pd.to_datetime(ph['end_time'],   utc=True)
df['timestamp']  = pd.to_datetime(df['timestamp'],  utc=True)

In [ ]:
df['timestamp']

In [ ]:
ph['end_time']

In [ ]:
ph['start_time']

In [ ]:
iv = pd.IntervalIndex.from_arrays(ph['start_time'], ph['end_time'], closed='both')

In [ ]:
iv

In [ ]:
idx = iv.get_indexer(df['timestamp'])

In [ ]:
idx

In [ ]:
df['phase_idx']     = idx
df['phase_name']    = np.where(idx >= 0, ph['phase_name'].to_numpy()[idx], None)
df['label']         = np.where(idx >= 0, ph['command'].to_numpy()[idx], 'discard')
df['phase_number']  = np.where(idx >= 0, ph['phase_number'].to_numpy()[idx], None)
df['step_number']   = np.where(idx >= 0, ph['step_number'].to_numpy()[idx], None)

In [ ]:
df['label'].value_counts()

In [ ]:
df['phase_name'].value_counts()

In [ ]:
print(f"step number - {len(df['step_number'].value_counts().keys())}")
print("step_number")
for name, value in df['step_number'].value_counts().items():
    print(f"{name}:\t{value}")

## Summary
Print a summary of the identified sequences for checking correctness.

In [ ]:
import pandas as pd
import numpy as np

df = df.sort_values('timestamp').reset_index(drop=True)

step_summary = (
    df.groupby('step_number', as_index=False)
      .agg(
          label        = ('label', 'first'),
          phase_name   = ('phase_name', 'first'),
          phase_number = ('phase_number', 'first'),
          num_flws     = ('timestamp', 'size'),
          start_time   = ('timestamp', 'min'),
          end_time     = ('timestamp', 'max')
      )
      .assign(duration=lambda g: g['end_time'] - g['start_time'])
)

step_summary['fps'] = (
    step_summary['num_flws'] /
    step_summary['duration'].dt.total_seconds().replace(0, np.nan)
)

label_summary = (
    step_summary.groupby('label', as_index=False)
               .agg(
                   avg_num_flws    = ('num_flws', 'mean'),
                   median_num_flws = ('num_flws', 'median'),
                   avg_duration    = ('duration', 'mean'),
                   median_duration = ('duration', 'median'),
                   total_steps = ('step_number', 'nunique'),
                   total_flws      = ('num_flws', 'sum'),
                   avg_fps         = ('fps', 'mean')
               )
)


In [ ]:
label_summary

In [ ]:
step_summary

## Remove Pauses 

In [ ]:
df = df[df['label'] != 'pause']
df = df[df['label'] != 'discard']

In [ ]:
df.shape

## Save labeled df to CSV

In [ ]:
df.to_csv(root + 'dollar_char_train_labeled.csv', index=False)